In [ ]:
import os
import torch
import torch.nn as nn
import torchaudio
import numpy as np
import cv2
from PIL import Image
from torchvision import transforms, models
import timm
from facenet_pytorch import MTCNN
from torch.cuda.amp import autocast

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"



In [ ]:
import torch
import torch.nn as nn
from torchvision import models
import timm
from torch.cuda.amp import autocast
from tqdm import tqdm

# same DEVICE as before
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ======================
# 1️⃣ DEFINE MODEL
# ======================
class CNN_Transformer_Fusion(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.cnn = models.efficientnet_b4(weights="IMAGENET1K_V1")
        self.cnn.classifier = nn.Identity()
        cnn_out = 1792

        self.transformer = timm.create_model("vit_base_patch16_224", pretrained=True)
        self.transformer.head = nn.Identity()
        trans_out = 768

        self.fusion = nn.Sequential(
            nn.Linear(cnn_out + trans_out, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        cnn_feat = self.cnn(x)
        trans_feat = self.transformer(x)
        fused = torch.cat([cnn_feat, trans_feat], dim=1)
        return self.fusion(fused)

# ======================
# 2️⃣ LOAD CHECKPOINT
# ======================
modelA = CNN_Transformer_Fusion(num_classes=2).to(DEVICE)
ckpt = torch.load("checkpoints/ckpt_epoch_10.pth", map_location=DEVICE)
modelA.load_state_dict(ckpt["model"])
modelA.eval()
print("✅ Loaded checkpoint from epoch 10")

In [ ]:
# Block 2: model
import torch.nn as nn
import torch.nn.functional as F

class SmallSpectroCNN(nn.Module):
    """Simple conv stack for spectrograms (input: [B,1,n_mels,T])"""
    def __init__(self, out_dim=256):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=(3,3), padding=1),  # [B,32,M,T]
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d((2,2)),  # downsample
            nn.Conv2d(32, 64, kernel_size=(3,3), padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d((2,2)),
            nn.Conv2d(64, 128, kernel_size=(3,3), padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )
        self.out_dim = out_dim
        # final projection done after folding

    def forward(self, x):
        # x: [B,1,M,T]
        return self.conv(x)  # [B, C, M', T']

class CNNTransformerAudio(nn.Module):
    def __init__(self, cnn_out_channels=128, d_model=256, nhead=8, num_layers=3, num_classes=2, dropout=0.2):
        super().__init__()
        self.cnn = SmallSpectroCNN()
        # after cnn, channels = 128 (from SmallSpectroCNN)
        self.conv_out_ch = 128
        # We'll collapse freq dimension (M') and keep time dimension T' as sequence length
        # project per-time features to d_model
        self.proj = nn.Conv1d(in_channels=self.conv_out_ch * ( (TARGET_N_MELS := 80) // 4 ),  # approximate M' after pooling / 4
                              out_channels=d_model, kernel_size=1)
        # Transformer encoder on sequence length T'
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=512, dropout=dropout, activation='relu')
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        # x: [B,1,M,T]
        B = x.shape[0]
        feat = self.cnn(x)             # [B, C, M', T']
        C, Mprime, Tprime = feat.shape[1], feat.shape[2], feat.shape[3]
        # Prepare per-time features: for each time index t, create vector of size (C * M')
        feat = feat.permute(0, 3, 1, 2).contiguous()  # [B, T', C, M']
        feat = feat.view(B, Tprime, C * Mprime)       # [B, T', C*M']
        # Transformer expects [S, B, D], so transpose:
        feat_t = feat.permute(1, 0, 2)                # [T', B, D]
        # project to d_model (using linear via 1d conv trick)
        feat_proj = self.proj(feat_t.permute(1,2,0))  # input [B, D, T'] -> out [B, d_model, T']
        feat_proj = feat_proj.permute(2,0,1)          # [T', B, d_model]
        trans_out = self.transformer(feat_proj)       # [T', B, d_model]
        # pool across time
        trans_out = trans_out.permute(1,2,0)          # [B, d_model, T']
        pooled = self.pool(trans_out).squeeze(-1)     # [B, d_model]
        logits = self.classifier(pooled)              # [B, num_classes]
        return logits


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

modelB = CNNTransformerAudio(num_classes=2).to(DEVICE)
ckptB = torch.load("asvspoof_checkpoints/best_asvspoof.pth", map_location=DEVICE)
modelB.load_state_dict(ckptB["model_state"])
modelB.eval()


In [ ]:
mtcnn = MTCNN(keep_all=False, device=DEVICE)
transform_img = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std=[0.229,0.224,0.225])
])

def extract_face_frames(video_path, max_frames=10):
    cap = cv2.VideoCapture(video_path)
    frames = []
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step = max(1, total // max_frames)
    idx = 0
    while cap.isOpened() and len(frames) < max_frames:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            break
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        boxes, _ = mtcnn.detect(Image.fromarray(rgb))
        if boxes is not None:
            x1,y1,x2,y2 = boxes[0].astype(int)
            pad = int(0.18 * max(x2-x1, y2-y1))
            y0, y3 = max(0, y1-pad), min(rgb.shape[0], y2+pad)
            x0, x3 = max(0, x1-pad), min(rgb.shape[1], x2+pad)
            face = rgb[y0:y3, x0:x3]
        else:
            face = rgb
        face = Image.fromarray(cv2.resize(face, (224,224)))
        frames.append(transform_img(face))
        idx += step
    cap.release()
    if len(frames)==0:
        raise ValueError("No frames extracted.")
    return torch.stack(frames)

def predict_visual(video_path):
    frames = extract_face_frames(video_path)
    with torch.no_grad(), autocast():
        logits = modelA(frames.to(DEVICE))
        probs = torch.softmax(logits, dim=1)
        avg_prob = probs.mean(dim=0)
    conf, pred = torch.max(avg_prob, dim=0)
    label = "Fake" if pred.item()==1 else "Real"
    return label, conf.item()

# -----------------------------
#  PREPROCESS — AUDIO
# -----------------------------
def predict_audio(video_path):
    try:
        waveform, sr = torchaudio.load(video_path)
    except:
        # extract audio track first
        import subprocess, tempfile
        tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
        cmd = ["ffmpeg", "-y", "-i", video_path, "-vn", "-ac", "1",
               "-ar", "16000", tmp.name]
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        waveform, sr = torchaudio.load(tmp.name)

    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)
    waveform = waveform / (waveform.abs().max() + 1e-6)

    mel = torchaudio.transforms.MelSpectrogram(
        sample_rate=16000, n_fft=512, hop_length=160, n_mels=80
    )(waveform)
    mel = torchaudio.transforms.AmplitudeToDB()(mel)
    mel = (mel - mel.mean()) / (mel.std(unbiased=False) + 1e-6)
    mel = mel.unsqueeze(0)  # [1,1,80,T]

    with torch.no_grad(), autocast():
        logits = modelB(mel.to(DEVICE))
        prob = torch.softmax(logits, dim=1).squeeze()
    conf, pred = torch.max(prob, dim=0)
    label = "Fake" if pred.item()==1 else "Real"
    return label, conf.item()


In [ ]:
if __name__ == "__main__":
    VIDEO_PATH = "datasets/FakeAVCeleb_v1.2/FakeAVCeleb_v1.2/FakeVideo-RealAudio/Asian (East)/men/id06470/00052_id00056_wavtolip.mp4"
    print(f"\n🧩 Running inference on: {VIDEO_PATH}")
    vis_label, vis_conf = predict_visual(VIDEO_PATH)
    #aud_label, aud_conf = predict_audio(VIDEO_PATH)

    print(f"\n🎥 Visual Prediction: {vis_label} ({vis_conf:.2f})")
    # print(f"🎧 Audio Prediction : {aud_label} ({aud_conf:.2f})")

    # # Optional final fusion decision
    # final = "Fake" if (vis_label=="Fake" or aud_label=="Fake") else "Real"
    # print(f"\n🧠 Final Decision: {final}")

In [ ]:
final_score = 0.6 * video_score + 0.4 * prob_fake
print(f"🔮 Fused fake probability: {final_score:.4f}")


In [ ]:
fused_prob = 0.6*video_prob + 0.4*audio_prob
fused_pred = "FAKE" if fused_prob >= 0.5 else "REAL"
print(f"🔮 Fused Result → {fused_pred}  (fake prob = {fused_prob:.4f})")

In [ ]:
import os
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# -----------------------------
# CONFIG
# -----------------------------
DATASET_BASE = Path("datasets/FakeAVCeleb_v1.2/FakeAVCeleb_v1.2")
OUTPUT_CSV = "fakeavceleb_metadata.csv"

LABEL_MAP = {
    "RealVideo-RealAudio": 0,  # Real
    "FakeVideo-RealAudio": 1,  # Fake
    "RealVideo-FakeAudio": 1,  # Fake
    "FakeVideo-FakeAudio": 1   # Fake
}

# -----------------------------
# SCAN FUNCTION
# -----------------------------
records = []

for folder, label in LABEL_MAP.items():
    folder_path = DATASET_BASE / folder
    if not folder_path.exists():
        print(f"[WARN] Missing folder: {folder_path}")
        continue

    for root, _, files in os.walk(folder_path):
        for f in files:
            if f.lower().endswith(".mp4"):
                full_path = os.path.join(root, f)
                records.append({
                    "path": full_path.replace("\\", "/"),
                    "label": label
                })

print(f"✅ Found {len(records)} total videos.")

# -----------------------------
# SAVE METADATA
# -----------------------------
df = pd.DataFrame(records)
df.to_csv(OUTPUT_CSV, index=False)
print(f"📄 Saved metadata to {OUTPUT_CSV}")
print(df.head())


In [ ]:
import pandas as pd

df = pd.read_csv("fakeavceleb_metadata.csv")

# Extract the top-level category (folder name)
df["category"] = df["path"].apply(
    lambda x: x.split("FakeAVCeleb_v1.2/")[-1].split("/")[0]
)

# Group and count
summary = df.groupby(["category", "label"]).size().reset_index(name="count")

print("\n📊 Video Distribution by Category:")
print(summary)

# Optional: show total by label
print("\n🧮 Total by label:")
print(df["label"].value_counts())


In [ ]:
import pandas as pd

# Load metadata
df = pd.read_csv("fakeavceleb_metadata.csv")

# Extract manipulation category (top-level folder)
df["category"] = df["path"].apply(lambda x: x.split("FakeAVCeleb_v1.2/")[-1].split("/")[0])

# Extract country (second-level folder)
df["country"] = df["path"].apply(lambda x: x.split("FakeAVCeleb_v1.2/")[-1].split("/")[1])

# Extract sex (third-level folder)
df["sex"] = df["path"].apply(lambda x: x.split("FakeAVCeleb_v1.2/")[-1].split("/")[2])

# --- 1️⃣ Category summary ---
cat_summary = df.groupby(["category", "label"]).size().reset_index(name="count")

# --- 2️⃣ Country summary ---
country_summary = df.groupby(["country", "label"]).size().reset_index(name="count")

# --- 3️⃣ Sex summary ---
sex_summary = df.groupby(["sex", "label"]).size().reset_index(name="count")

print("\n📊 Category-wise distribution:")
print(cat_summary)

print("\n🌍 Country-wise distribution:")
print(country_summary)

print("\n🧍 Sex-wise distribution:")
print(sex_summary)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

fig, axes = plt.subplots(1, 3, figsize=(18,5))

sns.barplot(data=cat_summary, x="category", y="count", hue="label", ax=axes[0])
axes[0].set_title("Category-wise Distribution")
axes[0].set_xlabel("Category")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis='x', rotation=45)

sns.barplot(data=country_summary, x="country", y="count", hue="label", ax=axes[1])
axes[1].set_title("Country-wise Distribution")
axes[1].tick_params(axis='x', rotation=45)

sns.barplot(data=sex_summary, x="sex", y="count", hue="label", ax=axes[2])
axes[2].set_title("Sex-wise Distribution")

plt.tight_layout()
plt.show()


In [ ]:
import os, cv2, torch, random, subprocess
import torchaudio
import numpy as np
from PIL import Image, ImageEnhance
from pathlib import Path
from tqdm import tqdm
from facenet_pytorch import MTCNN
from torchvision import transforms


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASE = Path("datasets/FakeAVCeleb_v1.2/FakeAVCeleb_v1.2")
OUT_VID = Path("preprocessed_fakeavceleb/video")
OUT_VID.mkdir(parents=True, exist_ok=True)

mtcnn = MTCNN(keep_all=False, device=DEVICE)
transform_img = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

def extract_faces(video_path, out_dir):
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step = max(1, total // 10)
    idx, saved = 0, 0
    while cap.isOpened() and saved < 10:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            break
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        boxes, _ = mtcnn.detect(Image.fromarray(rgb))
        if boxes is not None:
            x1, y1, x2, y2 = boxes[0].astype(int)
            pad = int(0.18 * max(x2 - x1, y2 - y1))
            y0, y3 = max(0, y1 - pad), min(rgb.shape[0], y2 + pad)
            x0, x3 = max(0, x1 - pad), min(rgb.shape[1], x2 + pad)
            face = rgb[y0:y3, x0:x3]
        else:
            face = rgb
        face = Image.fromarray(cv2.resize(face, (224, 224)))
        face_t = transform_img(face)
        torch.save(face_t, out_dir / f"frame_{saved:03d}.pt")
        saved += 1
        idx += step
    cap.release()
    return saved


In [ ]:
OUT_AUD = Path("preprocessed_fakeavceleb/audio")
OUT_AUD.mkdir(parents=True, exist_ok=True)

def extract_audio(video_path, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    wav_path = out_dir / "audio.wav"
    cmd = ["ffmpeg", "-y", "-i", str(video_path), "-vn", "-ac", "1", "-ar", "16000", str(wav_path)]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    waveform, sr = torchaudio.load(wav_path)
    waveform = waveform / (waveform.abs().max() + 1e-6)

    mel = torchaudio.transforms.MelSpectrogram(
        sample_rate=16000, n_fft=512, hop_length=160, n_mels=80
    )(waveform)
    mel = torchaudio.transforms.AmplitudeToDB()(mel)
    mel = (mel - mel.mean()) / (mel.std(unbiased=False) + 1e-6)

    torch.save(mel, out_dir / "mel.pt")
    os.remove(wav_path)
    return mel.shape[-1]


In [ ]:
OUT_BASE = Path("preprocessed_fakeavceleb")
OUT_BASE.mkdir(exist_ok=True, parents=True)

videos = [p for p in BASE.rglob("*.mp4")]
print(f"🎬 Found {len(videos)} videos")

for v in tqdm(videos):
    rel = v.relative_to(BASE)
    save_dir = OUT_BASE / rel.parent
    vid_dir = save_dir / v.stem
    vid_dir.mkdir(parents=True, exist_ok=True)

    # Extract video faces
    try:
        n_faces = extract_faces(v, vid_dir)
    except Exception as e:
        print(f"[Video ERR] {v}: {e}")
        continue

    # Extract audio spectrogram
    try:
        extract_audio(v, vid_dir)
    except Exception as e:
        print(f"[Audio ERR] {v}: {e}")
        continue


In [ ]:
# ==============================
#multimodel_data_preprocessing
# ==============================

import os
import csv
from pathlib import Path
from tqdm import tqdm

# ==============================
# CONFIG
# ==============================
PREPROCESSED_BASE = Path("preprocessed_fakeavceleb")
OUTPUT_CSV = "fakeavceleb_preprocessed_metadata.csv"

# Label map (same as before)
# 0 = Real, 1 = Fake
LABEL_MAP = {
    "RealVideo-RealAudio": 0,
    "FakeVideo-RealAudio": 1,
    "RealVideo-FakeAudio": 1,
    "FakeVideo-FakeAudio": 1
}

# ==============================
# COLLECT FOLDERS
# ==============================
rows = []
for category, label in LABEL_MAP.items():
    folder = PREPROCESSED_BASE / category
    if not folder.exists():
        print(f"⚠️ Missing: {folder}")
        continue

    # Each video folder has .pt frames and mel.pt
    for video_dir in tqdm(folder.rglob("*"), desc=f"{category}"):
        if video_dir.is_dir() and (video_dir / "mel.pt").exists():
            rows.append((str(video_dir), label))

print(f"\n✅ Total processed video folders found: {len(rows):,}")

# ==============================
# SAVE CSV
# ==============================
with open(OUTPUT_CSV, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["path", "label"])
    writer.writerows(rows)

print(f"📄 Metadata saved as: {OUTPUT_CSV}")


In [ ]:
import os
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchaudio
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm
import pandas as pd
import numpy as np

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 8
LR = 1e-4
EPOCHS = 10
SAVE_PATH = Path("modelC_checkpoints")
SAVE_PATH.mkdir(exist_ok=True)

print("Running on:", DEVICE)


In [ ]:
class FakeAVCelebDataset(Dataset):
    def __init__(self, csv_path):
        df = pd.read_csv(csv_path)
        self.data = df.values.tolist()

        # Image transform (same as Model A preprocessing)
        self.transform_img = T.Compose([
            T.Resize((224, 224)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        vid_path, label = self.data[idx]
        label = torch.tensor(label, dtype=torch.long)
        frames_dir = Path(vid_path)
        mel_path = frames_dir / "mel.pt"

        # ------------------------------
        # 🎞️ Load a few frames (1–5 evenly spaced)
        # ------------------------------
        frame_files = sorted(frames_dir.glob("frame_*.pt"))
        if len(frame_files) == 0:
            raise RuntimeError(f"No frames found in {frames_dir}")

        n = min(5, len(frame_files))
        select = np.linspace(0, len(frame_files) - 1, n, dtype=int)
        frames = []
        for i in select:
            try:
                frame = torch.load(frame_files[i], map_location="cpu")
                if isinstance(frame, torch.Tensor):
                    frames.append(frame)
                else:
                    raise TypeError(f"Frame is not tensor: {frame_files[i]}")
            except Exception as e:
                print(f"[WARN] Skipping frame {frame_files[i]}: {e}")
        imgs = torch.stack(frames)  # [N, 3, 224, 224]

        # ------------------------------
        # 🎧 Load mel-spectrogram
        # ------------------------------
        mel_loaded = torch.load(mel_path, map_location="cpu")

        # handle tensor/dict formats safely
        if isinstance(mel_loaded, dict):
            if "mel_spec" in mel_loaded:
                mel = mel_loaded["mel_spec"].float()
            elif "mel_db" in mel_loaded:
                mel = mel_loaded["mel_db"].float()
            elif "mel" in mel_loaded:
                mel = mel_loaded["mel"].float()
            else:
                raise KeyError(f"Unexpected dict keys in {mel_path}: {mel_loaded.keys()}")
        elif torch.is_tensor(mel_loaded):
            mel = mel_loaded.float()
        else:
            raise TypeError(f"Unexpected mel type in {mel_path}: {type(mel_loaded)}")

        # normalize mel (same as Model B preprocessing)
        mel = (mel - mel.mean()) / (mel.std(unbiased=False) + 1e-6)
        if mel.ndim == 2:
            mel = mel.unsqueeze(0)  # [1,80,T]
        elif mel.ndim == 3 and mel.shape[0] != 1:
            mel = mel.mean(0, keepdim=True)  # ensure single channel
        elif mel.ndim == 3 and mel.shape[0] == 1:
            pass  # already fine
        else:
            raise ValueError(f"Unexpected mel shape: {mel.shape}")


        return imgs, mel, label


In [ ]:
ds = FakeAVCelebDataset("fakeavceleb_preprocessed_metadata.csv")
imgs, mel, label = ds[0]
print("Mel shape:", mel.shape)


In [ ]:
# --- VIDEO ENCODER (Model A backbone) ---
class VideoEncoder(nn.Module):
    def __init__(self, modelA_path):
        super().__init__()
        from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
        base = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.features = base.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.out_dim = 1280
        if modelA_path and Path(modelA_path).exists():
            ckpt = torch.load(modelA_path, map_location="cpu")
            try: self.load_state_dict(ckpt["model_state"], strict=False)
            except: pass

    def forward(self, x):
        # x: [B,3,224,224]
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return x  # [B,1280]


# --- AUDIO ENCODER (Model B backbone) ---
class AudioEncoder(nn.Module):
    def __init__(self, modelB_path):
        super().__init__()
        from torch import nn
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, 1, 1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU()
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.out_dim = 128
        if modelB_path and Path(modelB_path).exists():
            ckpt = torch.load(modelB_path, map_location="cpu")
            try: self.load_state_dict(ckpt["model_state"], strict=False)
            except: pass

    def forward(self, x):
        # x: [B,1,80,T]
        x = self.cnn(x)
        x = self.pool(x).flatten(1)
        return x  # [B,128]


In [ ]:
class ModelC(nn.Module):
    def __init__(self, modelA_path, modelB_path, freeze_backbones=True):
        super().__init__()
        self.video_encoder = VideoEncoder(modelA_path)
        self.audio_encoder = AudioEncoder(modelB_path)

        if freeze_backbones:
            for p in self.video_encoder.parameters(): p.requires_grad = False
            for p in self.audio_encoder.parameters(): p.requires_grad = False

        fused_dim = self.video_encoder.out_dim + self.audio_encoder.out_dim
        self.fusion_head = nn.Sequential(
            nn.Linear(fused_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 2)
        )

    def forward(self, imgs, mels):
        # imgs: [Nf,3,224,224] ; mels: [1,1,80,T]
        Bv = imgs.shape[0]
        v_embs = [self.video_encoder(img.unsqueeze(0)) for img in imgs]
        v_feat = torch.stack(v_embs).mean(dim=0)  # [1,1280]

        a_feat = self.audio_encoder(mels)         # [1,128]
        fused = torch.cat([v_feat, a_feat], dim=1)
        out = self.fusion_head(fused)
        return out


In [ ]:
def train_epoch(model, loader, opt, loss_fn, scaler):
    model.train()
    total, correct, losses = 0, 0, []
    for imgs, mels, labels in tqdm(loader, desc="Train"):
        imgs = imgs.squeeze(0).to(DEVICE)
        mels = mels.to(DEVICE)
        labels = labels.to(DEVICE)

        opt.zero_grad()
        with autocast():
            logits = model(imgs, mels)
            loss = loss_fn(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()

        losses.append(loss.item())
        preds = torch.argmax(logits, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return np.mean(losses), correct/total




In [ ]:
import torch.nn.functional as F
from torch.cuda.amp import autocast

def evaluate_with_table(model, loader, n_samples=10000):
    """Show video/audio/fusion predictions neatly side by side."""
    model.eval()
    print(f"\n{'='*80}")
    print(f"{'GT':<8}{'Video':<20}{'Audio':<20}{'Fusion':<20}")
    print(f"{'-'*80}")

    shown = 0
    with torch.no_grad(), autocast():
        for imgs, mels, labels in loader:
            imgs = imgs.squeeze(0).to(DEVICE)
            mels = mels.to(DEVICE)
            label = labels.item()

            # 🎞️ Video embedding
            v_embs = [model.video_encoder(img.unsqueeze(0)) for img in imgs]
            v_feat = torch.stack(v_embs).mean(dim=0)

            # 🎧 Audio embedding
            a_feat = model.audio_encoder(mels)

            # 🔗 Fusion
            fused = torch.cat([v_feat, a_feat], dim=1)
            out = model.fusion_head(fused)
            prob = F.softmax(out, dim=1)
            conf, pred = torch.max(prob, dim=1)
            fusion_label = "Fake" if pred.item() == 1 else "Real"

            # Approximate visual/audio “fake likelihoods”
            v_score = torch.sigmoid(v_feat.mean()).item()
            a_score = torch.sigmoid(a_feat.mean()).item()

            v_label = "Fake" if v_score > 0.5 else "Real"
            a_label = "Fake" if a_score > 0.5 else "Real"
            gt_label = "Fake" if label == 1 else "Real"

            # ✅ Proper formatted print (fixed)
            print(f"{gt_label:<8}"
                  f"{v_label} ({v_score:.2f})".ljust(20) +
                  f"{a_label} ({a_score:.2f})".ljust(20) +
                  f"{fusion_label} ({conf.item():.2f})".ljust(20))

            shown += 1
            if shown >= n_samples:
                break

    print(f"{'='*80}\n")


In [ ]:
# Load dataset
ds = FakeAVCelebDataset("fakeavceleb_preprocessed_metadata.csv")
train_size = int(0.9 * len(ds))
test_size = len(ds) - train_size
train_ds, test_ds = torch.utils.data.random_split(ds, [train_size, test_size])
train_dl = DataLoader(train_ds, batch_size=1, shuffle=True)
test_dl  = DataLoader(test_ds,  batch_size=1, shuffle=False)

# Init model
model = ModelC(
    modelA_path="checkpoints/ckpt_epoch_10.pth",
    modelB_path="asvspoof_checkpoints/best/best_asvspoof.pth",
    freeze_backbones=True
).to(DEVICE)

opt = torch.optim.AdamW(model.fusion_head.parameters(), lr=LR)
loss_fn = nn.CrossEntropyLoss()
scaler = GradScaler()

best_acc = 0
for epoch in range(EPOCHS):
    tr_loss, tr_acc = train_epoch(model, train_dl, opt, loss_fn, scaler)
    te_loss, te_acc = evaluate(model, test_dl, loss_fn)
    print(f"Epoch {epoch+1}/{EPOCHS} — Train acc {tr_acc:.3f} | Test acc {te_acc:.3f}")

    if te_acc > best_acc:
        best_acc = te_acc
        torch.save(model.state_dict(), SAVE_PATH / "modelC_best.pth")
        print(f"✅ Saved best model: acc {best_acc:.3f}")


In [ ]:
# Make sure test_dl returns one sample at a time
test_small = torch.utils.data.Subset(test_ds, range(1))  # 5 samples for quick check
test_small_dl = DataLoader(test_small, batch_size=1, shuffle=False)

# Evaluate few samples visually
evaluate_with_table(model, test_small_dl, n_samples=1)


In [ ]:
from random import choice

# Sample one fake and one real folder
fake_sample = [x for x in ds.data if x[1] == 1]
real_sample = [x for x in ds.data if x[1] == 0]

mix = [choice(fake_sample), choice(real_sample)]
subset = torch.utils.data.Subset(ds, [ds.data.index(mix[0]), ds.data.index(mix[1])])
mix_dl = DataLoader(subset, batch_size=1)

evaluate_with_table(model, mix_dl, n_samples=2)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

# -----------------------------
#  Load ground truths & predictions
# -----------------------------
# (use your actual test loader results)
labels = np.array(all_labels)
preds = np.array(all_preds)

print(f"✅ Total evaluated samples: {len(labels)}")

# -----------------------------
#  Confusion Matrix
# -----------------------------
unique_classes = np.unique(np.concatenate([labels, preds]))
target_names = ["Real", "Fake"]

cm = confusion_matrix(labels, preds, labels=[0, 1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)

plt.figure(figsize=(6,5))
disp.plot(cmap="Purples", values_format="d")
plt.title("Confusion Matrix — Model C (FakeAVCeleb)")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.grid(False)
plt.show()

# -----------------------------
#  Classification Report
# -----------------------------
print("\n📄 Classification Report:")
print(classification_report(labels, preds, target_names=target_names, digits=4))

# -----------------------------
#  Load metadata
# -----------------------------
meta = pd.read_csv("fakeavceleb_preprocessed_metadata.csv")

# Ensure lengths match (adjust if your dataset split is smaller)
meta = meta.iloc[:len(labels)].copy()
meta["pred"] = preds
meta["true"] = labels

# -----------------------------
#  Category-wise Accuracy
# -----------------------------
category_acc = (
    meta.groupby("category")
        .apply(lambda x: (x["pred"] == x["true"]).mean())
        .reset_index(name="accuracy")
        .sort_values("accuracy", ascending=False)
)

plt.figure(figsize=(8,5))
sns.barplot(data=category_acc, x="accuracy", y="category", palette="cool")
plt.title("🎭 Category-wise Accuracy — Model C")
plt.xlabel("Accuracy")
plt.ylabel("Category")
plt.xlim(0,1)
plt.show()

print("\n🎭 Category-wise Accuracy:")
print(category_acc)

# -----------------------------
#  Country-wise Accuracy
# -----------------------------
country_acc = (
    meta.groupby("country")
        .apply(lambda x: (x["pred"] == x["true"]).mean())
        .reset_index(name="accuracy")
        .sort_values("accuracy", ascending=False)
)

plt.figure(figsize=(8,5))
sns.barplot(data=country_acc, x="accuracy", y="country", palette="viridis")
plt.title("🌍 Country-wise Accuracy — Model C")
plt.xlabel("Accuracy")
plt.ylabel("Country")
plt.xlim(0,1)
plt.show()

print("\n🌍 Country-wise Accuracy:")
print(country_acc)

# -----------------------------
#  Sex-wise Accuracy
# -----------------------------
sex_acc = (
    meta.groupby("sex")
        .apply(lambda x: (x["pred"] == x["true"]).mean())
        .reset_index(name="accuracy")
        .sort_values("accuracy", ascending=False)
)

plt.figure(figsize=(6,4))
sns.barplot(data=sex_acc, x="sex", y="accuracy", palette="magma")
plt.title("🧍 Sex-wise Accuracy — Model C")
plt.xlabel("Sex")
plt.ylabel("Accuracy")
plt.ylim(0,1)
plt.show()

print("\n🧍 Sex-wise Accuracy:")
print(sex_acc)

# -----------------------------
#  Country × Sex Heatmap
# -----------------------------
heatmap_df = (
    meta.groupby(["country", "sex"])
        .apply(lambda x: (x["pred"] == x["true"]).mean())
        .reset_index(name="accuracy")
        .pivot(index="country", columns="sex", values="accuracy")
)

plt.figure(figsize=(8,6))
sns.heatmap(heatmap_df, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("🌎 Accuracy Heatmap — Country × Sex (Model C)")
plt.xlabel("Sex")
plt.ylabel("Country")
plt.show()


In [ ]:
b = 10

In [ ]:
print(a+b)

In [6]:
#!/usr/bin/env python3
# full_pipeline_option3c_crossfusion.py
# Requires: torch, torchvision, torchaudio, facenet_pytorch (if using frame extraction),
# sklearn, tqdm, pandas, numpy, matplotlib (optional for plotting)

# ==============================
#modelC_multimodel_crossfusion_modeltrainig
# ==============================

import os
import random
import time
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import torchvision.transforms as T
from PIL import Image

# ---------------- USER CONFIG (adjust if needed) ----------------
METADATA_CSV = "fakeavceleb_preprocessed_metadata.csv"
MODEL_A_CKPT = "checkpoints/ckpt_epoch_10.pth"
MODEL_B_CKPT = "asvspoof_checkpoints/best/best_asvspoof.pth"
OUTPUT_DIR = Path("modelC_option3c_crossfusion")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
BATCH_SIZE = 8
NUM_WORKERS = 0
EPOCHS = 20
LR_FUSION = 1e-4
LR_ENCODER = 1e-5
FREEZE_EPOCHS = 3      # number of epochs to keep backbones frozen
NUM_FRAMES = 5         # frames per video
MEL_MAX_LEN = None     # None => pad to batch max
CONTRASTIVE_WEIGHT = 0.35
USE_WEIGHTED_SAMPLER = True
PRINT_EVERY = 50

# reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ---------------- Dataset ----------------
class FakeAVCelebDataset(Dataset):
    """
    Expects METADATA_CSV with columns: path,label
    Each path -> folder containing:
      - frame_000... .pt (torch tensor  [3,224,224]) OR frame_*.jpg
      - mel.pt  (tensor [1,80,T] or dict containing mel_spec/mel_db)
    Returns:
      frames: Tensor [Nf,3,224,224]
      mel: Tensor [1,80,T]
      label: int
    """
    def __init__(self, csv_path, num_frames=NUM_FRAMES, transform=None):
        self.df = pd.read_csv(csv_path)
        if list(self.df.columns[:2]) != ["path", "label"]:
            self.df = self.df.iloc[:, :2]
            self.df.columns = ["path", "label"]
        self.items = self.df.values.tolist()
        self.num_frames = num_frames
        self.transform = transform or T.Compose([
            T.Resize((224,224)),
            T.ToTensor()
        ])

    def __len__(self):
        return len(self.items)

    def _load_frames(self, folder):
        folder = Path(folder)
        pts = sorted(folder.glob("frame_*.pt"))
        if len(pts) == 0:
            imgs = sorted(folder.glob("*.jpg"))
            if len(imgs) == 0:
                raise RuntimeError(f"No frames found in {folder}")
            # load images and transform
            files = imgs
            frames = []
            idxs = np.linspace(0, len(files)-1, min(self.num_frames, len(files)), dtype=int)
            for i in idxs:
                im = Image.open(files[i]).convert("RGB")
                frames.append(self.transform(im))
            return torch.stack(frames)  # [Nf,3,224,224]
        else:
            files = pts
            n = min(self.num_frames, len(files))
            idxs = np.linspace(0, len(files)-1, n, dtype=int)
            frames = []
            for i in idxs:
                t = torch.load(files[i], map_location="cpu")
                if not torch.is_tensor(t):
                    raise TypeError(f"Frame {files[i]} doesn't contain tensor.")
                frames.append(t)
            return torch.stack(frames)

    def _load_mel(self, mel_path):
        mel_loaded = torch.load(mel_path, map_location="cpu")
        if isinstance(mel_loaded, dict):
            if "mel_spec" in mel_loaded:
                mel = mel_loaded["mel_spec"].float()
            elif "mel_db" in mel_loaded:
                mel = mel_loaded["mel_db"].float()
            elif "mel" in mel_loaded:
                mel = mel_loaded["mel"].float()
            else:
                # try first tensor-like value
                first = next(iter(mel_loaded.values()))
                mel = torch.tensor(first).float()
        elif torch.is_tensor(mel_loaded):
            mel = mel_loaded.float()
        else:
            raise TypeError(f"Unsupported mel content at {mel_path}: {type(mel_loaded)}")
        if mel.dim() == 2:
            mel = mel.unsqueeze(0)
        # normalize per-sample
        mel = (mel - mel.mean()) / (mel.std(unbiased=False) + 1e-6)
        return mel

    def __getitem__(self, idx):
        folder, label = self.items[idx]
        frames = self._load_frames(folder)
        mel_path = Path(folder) / "mel.pt"
        if not mel_path.exists():
            raise RuntimeError(f"Missing mel.pt for {folder}")
        mel = self._load_mel(mel_path)
        return frames, mel, torch.tensor(int(label), dtype=torch.long)

def collate_variable(batch, max_mel_len=MEL_MAX_LEN):
    B = len(batch)
    Nf = batch[0][0].shape[0]
    C,H,W = batch[0][0].shape[1:]
    frames = torch.stack([b[0] for b in batch], dim=0)  # [B,Nf,3,H,W]
    Ts = [b[1].shape[-1] for b in batch]
    Tmax = max(Ts) if max_mel_len is None else min(max(Ts), max_mel_len)
    mels = torch.zeros(B, 1, batch[0][1].shape[1], Tmax, dtype=torch.float32)
    for i, b in enumerate(batch):
        t = b[1].shape[-1]
        copy_t = min(t, Tmax)
        mels[i,:, :, :copy_t] = b[1][:, :, :copy_t]
    labels = torch.stack([b[2] for b in batch])
    return frames, mels, labels

# ---------------- Load datasets & sampler ----------------
print("Loading dataset...")
ds = FakeAVCelebDataset(METADATA_CSV, num_frames=NUM_FRAMES)
labels = [int(x[1]) for x in ds.items]
print("Samples:", len(ds), "Label counts:", pd.Series(labels).value_counts().to_dict())

# split 90/10
idxs = np.arange(len(ds))
np.random.shuffle(idxs)
train_idx = idxs[: int(0.9 * len(idxs))]
val_idx = idxs[int(0.9 * len(idxs)):]

if USE_WEIGHTED_SAMPLER:
    vals, counts = np.unique(labels, return_counts=True)
    class_weights = {int(v): len(labels)/c for v,c in zip(vals,counts)}
    sample_weights = [class_weights[int(labels[i])] for i in range(len(labels))]
    train_sampler = WeightedRandomSampler([sample_weights[i] for i in train_idx], num_samples=len(train_idx), replacement=True)
    train_loader = DataLoader(torch.utils.data.Subset(ds, train_idx), batch_size=BATCH_SIZE,
                              sampler=train_sampler,
                              collate_fn=collate_variable, num_workers=NUM_WORKERS, pin_memory=True)
else:
    train_loader = DataLoader(torch.utils.data.Subset(ds, train_idx), batch_size=BATCH_SIZE,
                              shuffle=True, collate_fn=collate_variable, num_workers=NUM_WORKERS, pin_memory=True)

val_loader = DataLoader(torch.utils.data.Subset(ds, val_idx), batch_size=BATCH_SIZE,
                        shuffle=False, collate_fn=collate_variable, num_workers=NUM_WORKERS)

# ---------------- Encoders (ModelA & ModelB) ----------------
# For Model A: small wrapper around EfficientNet-B4 + ViT fusion from your previous code.
from torchvision import models
import timm

class ModelA_Visual(nn.Module):
    def __init__(self, ckpt_path=None, device="cpu", embed_dim=256):
        super().__init__()
        # EfficientNet-B4 backbone
        self.cnn = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.IMAGENET1K_V1)
        self.cnn.classifier = nn.Identity()
        cnn_out = 1792
        # ViT
        self.vit = timm.create_model("vit_base_patch16_224", pretrained=True)
        self.vit.head = nn.Identity()
        trans_out = 768
        # simple proj to embed_dim
        self.proj = nn.Linear(cnn_out + trans_out, embed_dim)
        self.embed_dim = embed_dim
        if ckpt_path and Path(ckpt_path).exists():
            try:
                ckpt = torch.load(ckpt_path, map_location=device)
                state = ckpt.get("model_state", ckpt.get("model", ckpt))
                # attempt to load by name; we will be forgiving
                self.load_state_dict(state, strict=False)
                print(f"[INFO] ModelA loaded from {ckpt_path} (partial load allowed).")
            except Exception as e:
                print(f"[WARN] Could not load ModelA ckpt: {e}")

    def forward(self, frames):
        # frames: [B,Nf,3,224,224]
        B, Nf = frames.shape[0], frames.shape[1]
        x = frames.view(B * Nf, 3, 224, 224).to(next(self.parameters()).device)
        cnn_feat = self.cnn(x)            # [B*Nf, cnn_out]
        vit_feat = self.vit(x)            # [B*Nf, trans_out]
        fused = torch.cat([cnn_feat, vit_feat], dim=1)
        fused = fused.view(B, Nf, -1).mean(1)   # mean pool frames -> [B, cnn+vit]
        emb = self.proj(fused)            # [B, embed_dim]
        return emb

class ModelB_Audio(nn.Module):
    def __init__(self, ckpt_path=None, device="cpu", embed_dim=256):
        super().__init__()
        # small spectrogram CNN + transformer like your modelB
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, 1, 1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU()
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Linear(128, embed_dim)
        self.embed_dim = embed_dim
        if ckpt_path and Path(ckpt_path).exists():
            try:
                ckpt = torch.load(ckpt_path, map_location=device)
                state = ckpt.get("model_state", ckpt.get("model", ckpt))
                self.load_state_dict(state, strict=False)
                print(f"[INFO] ModelB loaded from {ckpt_path} (partial load allowed).")
            except Exception as e:
                print(f"[WARN] Could not load ModelB ckpt: {e}")

    def forward(self, mels):
        # mels: [B,1,80,T]
        x = self.cnn(mels)     # [B, C, h, w]
        x = self.pool(x).flatten(1)   # [B, C]
        emb = self.proj(x)     # [B, embed_dim]
        return emb

# ---------------- Cross-Attention Fusion Head ----------------
class CrossAttentionFusion(nn.Module):
    def __init__(self, dim=256, heads=4, dropout=0.1):
        super().__init__()
        self.dim = dim
        self.attn_va = nn.MultiheadAttention(embed_dim=dim, num_heads=heads, batch_first=True, dropout=dropout)
        self.attn_av = nn.MultiheadAttention(embed_dim=dim, num_heads=heads, batch_first=True, dropout=dropout)
        self.norm_v = nn.LayerNorm(dim)
        self.norm_a = nn.LayerNorm(dim)
        self.classifier = nn.Sequential(
            nn.Linear(dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 2)
        )

    def forward(self, v_feat, a_feat):
        v_tok = v_feat.unsqueeze(1)
        a_tok = a_feat.unsqueeze(1)
        v2a, _ = self.attn_va(v_tok, a_tok, a_tok)  
        a2v, _ = self.attn_av(a_tok, v_tok, v_tok)
        v_out = self.norm_v(v_tok + v2a).squeeze(1)  
        a_out = self.norm_a(a_tok + a2v).squeeze(1)
        fused = (v_out + a_out) / 2.0                
        logits = self.classifier(fused)
        return logits, v_out, a_out

# ---------------- Full AV Model ----------------
class AVFusionModel(nn.Module):
    def __init__(self, modelA, modelB, fusion_head):
        super().__init__()
        self.modelA = modelA
        self.modelB = modelB
        self.fusion = fusion_head

    def forward(self, frames, mels):
        v = self.modelA(frames)   
        a = self.modelB(mels)     
        logits, v_emb, a_emb = self.fusion(v, a)
        return logits, v_emb, a_emb

# ---------------- Build models ----------------
print("Building models...")
modelA = ModelA_Visual(ckpt_path=MODEL_A_CKPT, device=DEVICE, embed_dim=256).to(DEVICE)
modelB = ModelB_Audio(ckpt_path=MODEL_B_CKPT, device=DEVICE, embed_dim=256).to(DEVICE)
fusion_head = CrossAttentionFusion(dim=256, heads=4).to(DEVICE)
avmodel = AVFusionModel(modelA, modelB, fusion_head).to(DEVICE)

# ---------------- Freeze schedule ----------------
def set_requires_grad(module, requires):
    for p in module.parameters():
        p.requires_grad = requires

# Freeze encoders initially
set_requires_grad(modelA, False)
set_requires_grad(modelB, False)
# fusion head trainable
set_requires_grad(fusion_head, True)

# ---------------- Losses / Optimizers ----------------
# compute class weights
vals, counts = np.unique(labels, return_counts=True)
cw = {int(v): len(labels)/c for v,c in zip(vals,counts)}
weight_tensor = torch.tensor([cw.get(0,1.0), cw.get(1,1.0)], dtype=torch.float).to(DEVICE)
ce_loss_fn = nn.CrossEntropyLoss(weight=weight_tensor)

# optimizer: different lrs for fusion and encoders (encoders frozen initially)
opt = torch.optim.AdamW([
    {"params": fusion_head.parameters(), "lr": LR_FUSION},
    {"params": modelA.parameters(), "lr": LR_ENCODER},
    {"params": modelB.parameters(), "lr": LR_ENCODER},
], weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

# ---------------- Contrastive loss helper ----------------
def contrastive_loss(v_emb, a_emb, labels, margin=0.5):
    cos = F.cosine_similarity(v_emb, a_emb, dim=1) 
    pos_mask = (labels == 0).float()    
    neg_mask = (labels == 1).float()    
    pos_loss = pos_mask * (1.0 - cos)
    neg_loss = neg_mask * F.relu(cos - margin)
    loss = (pos_loss + neg_loss).mean()
    return loss

# ---------------- Train / Eval ----------------
def train_one_epoch(epoch):
    avmodel.train()
    running_loss = 0.0
    running_ce = 0.0
    running_con = 0.0
    total = 0
    correct = 0
    for i, (frames, mels, labels_batch) in enumerate(tqdm(train_loader, desc=f"Train E{epoch}")):
        frames = frames.to(DEVICE).float()
        mels = mels.to(DEVICE).float()
        labels_batch = labels_batch.to(DEVICE)
        # Optional: random feature interchange augmentation (swap audio in some samples)
        if random.random() < 0.15:
            # swap audio between two random items in batch
            j = torch.randperm(frames.size(0))
            mels = mels[j]

        opt.zero_grad()
        logits, v_emb, a_emb = avmodel(frames, mels)
        ce = ce_loss_fn(logits, labels_batch)
        con = contrastive_loss(v_emb, a_emb, labels_batch)
        loss = ce + CONTRASTIVE_WEIGHT * con
        loss.backward()
        opt.step()

        running_loss += float(loss.item()) * labels_batch.size(0)
        running_ce += float(ce.item()) * labels_batch.size(0)
        running_con += float(con.item()) * labels_batch.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels_batch).sum().item()
        total += labels_batch.size(0)

        if (i+1) % PRINT_EVERY == 0:
            print(f"[E{epoch}] iter {i+1} loss {running_loss/total:.4f} ce {running_ce/total:.4f} con {running_con/total:.4f}")

    acc = correct / max(1, total)
    avg_loss = running_loss / max(1, total)
    return avg_loss, acc

@torch.no_grad()
def validate(epoch):
    avmodel.eval()
    ys, yps, probs = [], [], []
    total, correct = 0, 0
    for frames, mels, labels_batch in tqdm(val_loader, desc=f"Val E{epoch}"):
        frames = frames.to(DEVICE).float()
        mels = mels.to(DEVICE).float()
        labels_batch = labels_batch.to(DEVICE)
        logits, v_emb, a_emb = avmodel(frames, mels)
        prob = F.softmax(logits, dim=1)[:,1]
        preds = logits.argmax(dim=1)
        ys.extend(labels_batch.cpu().numpy().tolist())
        yps.extend(preds.cpu().numpy().tolist())
        probs.extend(prob.cpu().numpy().tolist())
        correct += (preds == labels_batch).sum().item()
        total += labels_batch.size(0)
    acc = correct / max(1, total)
    return np.array(ys), np.array(yps), np.array(probs), acc

# ---------------- Training loop ----------------
best_acc = -1.0
print("Start training on device:", DEVICE)

for epoch in range(1, EPOCHS + 1):
   
    if epoch == FREEZE_EPOCHS + 1:
        print(f"[SCHEDULE] Unfreezing encoders at epoch {epoch}")
        set_requires_grad(modelA, True)
        set_requires_grad(modelB, True)
        set_requires_grad(fusion_head, True)
        opt = torch.optim.AdamW([
            {"params": fusion_head.parameters(), "lr": LR_FUSION},
            {"params": modelA.parameters(), "lr": LR_ENCODER},
            {"params": modelB.parameters(), "lr": LR_ENCODER},
        ], weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, EPOCHS - epoch + 1))

    train_loss, train_acc = train_one_epoch(epoch)
    ys, yps, probs, val_acc = validate(epoch)
    scheduler.step()

    print(f"Epoch {epoch}/{EPOCHS} — train_loss {train_loss:.4f} train_acc {train_acc:.4f} val_acc {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({
            "epoch": epoch,
            "model_state": avmodel.state_dict(),
            "modelA_state": modelA.state_dict(),
            "modelB_state": modelB.state_dict(),
            "fusion_state": fusion_head.state_dict(),
            "optimizer": opt.state_dict(),
            "best_acc": best_acc
        }, OUTPUT_DIR / "modelC_crossfusion_best.pth")
        print(f"Saved best model (val_acc {best_acc:.4f}) to {OUTPUT_DIR/'modelC_crossfusion_best.pth'}")

    if epoch % 5 == 0:
        torch.save({
            "epoch": epoch,
            "model_state": avmodel.state_dict(),
            "optimizer": opt.state_dict(),
        }, OUTPUT_DIR / f"modelC_crossfusion_epoch{epoch}.pth")

# ---------------- Final evaluation & summary ----------------
print("Final evaluation on validation set...")
ys, yps, probs, val_acc = validate("final")
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
print("Val acc:", val_acc)
print(classification_report(ys, yps, target_names=["Real","Fake"], digits=4))
cm = confusion_matrix(ys, yps)
print("Confusion matrix:\n", cm)
try:
    auc = roc_auc_score(ys, probs)
    print("ROC AUC:", auc)
except Exception as e:
    print("Could not compute AUC:", e)

# save predictions
pd.DataFrame({"true": ys, "pred": yps, "prob_fake": probs}).to_csv(OUTPUT_DIR / "predictions_val.csv", index=False)
print("Saved predictions to", OUTPUT_DIR / "predictions_val.csv")
print("Training finished. Best val acc:", best_acc)


Loading dataset...
Samples: 21544 Label counts: {1: 21044, 0: 500}
Building models...


C:\Users\admin\AppData\Local\Temp\ipykernel_30336\1509094331.py:197: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location=device)


[INFO] ModelA loaded from checkpoints/ckpt_epoch_10.pth (partial load allowed).


C:\Users\admin\AppData\Local\Temp\ipykernel_30336\1509094331.py:230: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location=device)


[WARN] Could not load ModelB ckpt: Error(s) in loading state_dict for ModelB_Audio:
	size mismatch for proj.weight: copying a param with shape torch.Size([256, 2560, 1]) from checkpoint, the shape in current model is torch.Size([256, 128]).
Start training on device: cuda


...

[E20] iter 2400 loss 0.0053 ce 0.0039 con 0.0040


Val E20: 100%|███████████████████████████████████████████████████████████████████████| 270/270 [03:03<00:00,  1.48it/s]


Epoch 20/20 — train_loss 0.0053 train_acc 0.9866 val_acc 0.9759
Final evaluation on validation set...


Val Efinal: 100%|████████████████████████████████████████████████████████████████████| 270/270 [02:52<00:00,  1.57it/s]

Val acc: 0.9758700696055684
              precision    recall  f1-score   support

        Real     0.4516    0.9767    0.6176        43
        Fake     0.9995    0.9759    0.9875      2112

    accuracy                         0.9759      2155
   macro avg     0.7256    0.9763    0.8026      2155
weighted avg     0.9886    0.9759    0.9802      2155

Confusion matrix:
 [[  42    1]
 [  51 2061]]
ROC AUC: 0.9931179527836504
Saved predictions to modelC_option3c_crossfusion\predictions_val.csv
Training finished. Best val acc: 0.9777262180974478


In [12]:
#modelA_loder

class CNN_Transformer_Fusion(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.cnn = models.efficientnet_b4(weights="IMAGENET1K_V1")
        self.cnn.classifier = nn.Identity()
        cnn_out = 1792

        self.transformer = timm.create_model("vit_base_patch16_224", pretrained=True)
        self.transformer.head = nn.Identity()
        trans_out = 768

        self.fusion = nn.Sequential(
            nn.Linear(cnn_out + trans_out, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        cnn_feat = self.cnn(x)
        trans_feat = self.transformer(x)
        fused = torch.cat([cnn_feat, trans_feat], dim=1)
        return self.fusion(fused)

#modelB_loder

class SmallSpectroCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.conv(x)



class CNNTransformerAudio(nn.Module):
    def __init__(self, num_classes=2, target_n_mels=80):
        super().__init__()
        self.cnn = SmallSpectroCNN()
        self.conv_out_ch = 128
        self.target_n_mels = target_n_mels

        self.proj = nn.Conv1d(
            in_channels=self.conv_out_ch * (self.target_n_mels // 4),
            out_channels=256,
            kernel_size=1
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=256,
            nhead=8,
            dim_feedforward=512,
            dropout=0.2,
            activation='relu',
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=3)
        self.pool = nn.AdaptiveAvgPool1d(1)

        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        feat = self.cnn(x)             # [B, 128, mel/4, T/4]
        B, C, M, T = feat.shape
        feat = feat.view(B, C * M, T)
        feat = self.proj(feat).permute(0, 2, 1)
        trans_out = self.transformer(feat)
        pooled = self.pool(trans_out.transpose(1, 2)).squeeze(-1)
        return self.classifier(pooled)



In [ ]:
#!/usr/bin/env python3
# full_pipeline_option3c_crossfusion.py
# Requires: torch, torchvision, torchaudio, facenet_pytorch (if using frame extraction),
# sklearn, tqdm, pandas, numpy, matplotlib (optional for plotting)

import os
import random
import time
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
from numpy.core.multiarray import scalar, dtype
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import torchvision.transforms as T
from PIL import Image

# ---------------- USER CONFIG (adjust if needed) ----------------
METADATA_CSV = "fakeavceleb_preprocessed_metadata.csv"
MODEL_A_CKPT = "checkpoints/ckpt_epoch_10.pth"
MODEL_B_CKPT = "asvspoof_checkpoints/best/best_asvspoof.pth"
OUTPUT_DIR = Path("modelC_option3c_crossfusion")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
BATCH_SIZE = 8
NUM_WORKERS = 0
EPOCHS = 20
LR_FUSION = 1e-4
LR_ENCODER = 1e-5
FREEZE_EPOCHS = 3      # number of epochs to keep backbones frozen
NUM_FRAMES = 5         # frames per video
MEL_MAX_LEN = None     # None => pad to batch max
CONTRASTIVE_WEIGHT = 0.35
USE_WEIGHTED_SAMPLER = True
PRINT_EVERY = 50

# reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ---------------- Dataset ----------------
class FakeAVCelebDataset(Dataset):
    """
    Expects METADATA_CSV with columns: path,label
    Each path -> folder containing:
      - frame_000... .pt (torch tensor  [3,224,224]) OR frame_*.jpg
      - mel.pt  (tensor [1,80,T] or dict containing mel_spec/mel_db)
    Returns:
      frames: Tensor [Nf,3,224,224]
      mel: Tensor [1,80,T]
      label: int
    """
    def __init__(self, csv_path, num_frames=NUM_FRAMES, transform=None):
        self.df = pd.read_csv(csv_path)
        if list(self.df.columns[:2]) != ["path", "label"]:
            self.df = self.df.iloc[:, :2]
            self.df.columns = ["path", "label"]
        self.items = self.df.values.tolist()
        self.num_frames = num_frames
        self.transform = transform or T.Compose([
            T.Resize((224,224)),
            T.ToTensor()
        ])

    def __len__(self):
        return len(self.items)

    def _load_frames(self, folder):
        folder = Path(folder)
        pts = sorted(folder.glob("frame_*.pt"))
        if len(pts) == 0:
            imgs = sorted(folder.glob("*.jpg"))
            if len(imgs) == 0:
                raise RuntimeError(f"No frames found in {folder}")
            # load images and transform
            files = imgs
            frames = []
            idxs = np.linspace(0, len(files)-1, min(self.num_frames, len(files)), dtype=int)
            for i in idxs:
                im = Image.open(files[i]).convert("RGB")
                frames.append(self.transform(im))
            return torch.stack(frames)  # [Nf,3,224,224]
        else:
            files = pts
            n = min(self.num_frames, len(files))
            idxs = np.linspace(0, len(files)-1, n, dtype=int)
            frames = []
            for i in idxs:
                t = torch.load(files[i], map_location="cpu", weights_only=True)
                if not torch.is_tensor(t):
                    raise TypeError(f"Frame {files[i]} doesn't contain tensor.")
                frames.append(t)
            return torch.stack(frames)

    def _load_mel(self, mel_path):
        mel_loaded = torch.load(mel_path, map_location="cpu", weights_only=True)
        if isinstance(mel_loaded, dict):
            if "mel_spec" in mel_loaded:
                mel = mel_loaded["mel_spec"].float()
            elif "mel_db" in mel_loaded:
                mel = mel_loaded["mel_db"].float()
            elif "mel" in mel_loaded:
                mel = mel_loaded["mel"].float()
            else:
                # try first tensor-like value
                first = next(iter(mel_loaded.values()))
                mel = torch.tensor(first).float()
        elif torch.is_tensor(mel_loaded):
            mel = mel_loaded.float()
        else:
            raise TypeError(f"Unsupported mel content at {mel_path}: {type(mel_loaded)}")
        if mel.dim() == 2:
            mel = mel.unsqueeze(0)
        # normalize per-sample
        mel = (mel - mel.mean()) / (mel.std(unbiased=False) + 1e-6)
        return mel

    def __getitem__(self, idx):
        folder, label = self.items[idx]
        frames = self._load_frames(folder)
        mel_path = Path(folder) / "mel.pt"
        if not mel_path.exists():
            raise RuntimeError(f"Missing mel.pt for {folder}")
        mel = self._load_mel(mel_path)
        return frames, mel, torch.tensor(int(label), dtype=torch.long)

def collate_variable(batch, max_mel_len=MEL_MAX_LEN):
    B = len(batch)
    Nf = batch[0][0].shape[0]
    C,H,W = batch[0][0].shape[1:]
    frames = torch.stack([b[0] for b in batch], dim=0)  # [B,Nf,3,H,W]
    Ts = [b[1].shape[-1] for b in batch]
    Tmax = max(Ts) if max_mel_len is None else min(max(Ts), max_mel_len)
    mels = torch.zeros(B, 1, batch[0][1].shape[1], Tmax, dtype=torch.float32)
    for i, b in enumerate(batch):
        t = b[1].shape[-1]
        copy_t = min(t, Tmax)
        mels[i,:, :, :copy_t] = b[1][:, :, :copy_t]
    labels = torch.stack([b[2] for b in batch])
    return frames, mels, labels

# ---------------- Load datasets & sampler ----------------
print("Loading dataset...")
ds = FakeAVCelebDataset(METADATA_CSV, num_frames=NUM_FRAMES)
labels = [int(x[1]) for x in ds.items]
print("Samples:", len(ds), "Label counts:", pd.Series(labels).value_counts().to_dict())

# split 90/10
idxs = np.arange(len(ds))
np.random.shuffle(idxs)
train_idx = idxs[: int(0.9 * len(idxs))]
val_idx = idxs[int(0.9 * len(idxs)):]

if USE_WEIGHTED_SAMPLER:
    vals, counts = np.unique(labels, return_counts=True)
    class_weights = {int(v): len(labels)/c for v,c in zip(vals,counts)}
    sample_weights = [class_weights[int(labels[i])] for i in range(len(labels))]
    train_sampler = WeightedRandomSampler([sample_weights[i] for i in train_idx], num_samples=len(train_idx), replacement=True)
    train_loader = DataLoader(torch.utils.data.Subset(ds, train_idx), batch_size=BATCH_SIZE,
                              sampler=train_sampler,
                              collate_fn=collate_variable, num_workers=NUM_WORKERS, pin_memory=True)
else:
    train_loader = DataLoader(torch.utils.data.Subset(ds, train_idx), batch_size=BATCH_SIZE,
                              shuffle=True, collate_fn=collate_variable, num_workers=NUM_WORKERS, pin_memory=True)

val_loader = DataLoader(torch.utils.data.Subset(ds, val_idx), batch_size=BATCH_SIZE,
                        shuffle=False, collate_fn=collate_variable, num_workers=NUM_WORKERS)

# ---------------- Encoders (ModelA & ModelB) ----------------
# For Model A: small wrapper around EfficientNet-B4 + ViT fusion from your previous code.
from torchvision import models
import timm

# ---------------- ModelA wrapper (visual) ----------------
class ModelA_Visual(nn.Module):
    def __init__(self, ckpt_path=None, device="cpu", embed_dim=256):
        super().__init__()
        # Use your CNN+Transformer fusion
        self.model = CNN_Transformer_Fusion(num_classes=embed_dim).to(device)
        self.embed_dim = embed_dim

        if ckpt_path and Path(ckpt_path).exists():
            try:
                ckpt = torch.load(ckpt_path, map_location=device)
                state = ckpt.get("model_state", ckpt.get("model", ckpt))
                
                # Remove fusion/classifier weights to avoid size mismatch
                for k in list(state.keys()):
                    if "fusion" in k:
                        state.pop(k)
                
                self.model.load_state_dict(state, strict=False)
                print(f"[INFO] ModelA loaded pretrained CNN/Transformer weights from {ckpt_path}")
            except Exception as e:
                print(f"[WARN] Could not load ModelA ckpt: {e}")

    def forward(self, frames):
        B, Nf = frames.shape[0], frames.shape[1]
        x = frames.view(B * Nf, 3, 224, 224).to(next(self.model.parameters()).device)
        fused = self.model(x)  # [B*Nf, embed_dim]
        fused = fused.view(B, Nf, -1).mean(1)
        return fused  # [B, embed_dim]


# ---------------- ModelB wrapper (audio) ----------------
class ModelB_Audio(nn.Module):
    def __init__(self, ckpt_path=None, device="cpu", embed_dim=256):
        super().__init__()
        # Use your CNN+Transformer audio model
        self.model = CNNTransformerAudio(num_classes=embed_dim).to(device)
        self.embed_dim = embed_dim

        if ckpt_path and Path(ckpt_path).exists():
            try:
                torch.serialization.add_safe_globals([np.dtype, np.float32, np.float64])
                ckpt = torch.load(ckpt_path, map_location=device, weights_only=True)
                state = ckpt.get("model_state", ckpt.get("model", ckpt))
                
                # Remove classifier weights to avoid size mismatch
                for k in list(state.keys()):
                    if "classifier" in k:
                        state.pop(k)
                
                self.model.load_state_dict(state, strict=False)
                print(f"[INFO] ModelB loaded pretrained CNN/Transformer weights from {ckpt_path}")
            except Exception as e:
                print(f"[WARN] Could not load ModelB ckpt: {e}")

    def forward(self, mels):
        return self.model(mels)  # [B, embed_dim]

# ---------------- Fusion ----------------
class CrossAttentionFusion(nn.Module):
    def __init__(self, dim=256, heads=4, dropout=0.1):
        super().__init__()
        self.attn_va = nn.MultiheadAttention(embed_dim=dim, num_heads=heads, batch_first=True, dropout=dropout)
        self.attn_av = nn.MultiheadAttention(embed_dim=dim, num_heads=heads, batch_first=True, dropout=dropout)
        self.norm_v = nn.LayerNorm(dim)
        self.norm_a = nn.LayerNorm(dim)
        self.classifier = nn.Sequential(
            nn.Linear(dim, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 2)
        )

    def forward(self, v_feat, a_feat):
        v_tok = v_feat.unsqueeze(1)
        a_tok = a_feat.unsqueeze(1)
        v2a, _ = self.attn_va(v_tok, a_tok, a_tok)
        a2v, _ = self.attn_av(a_tok, v_tok, v_tok)
        v_out = self.norm_v(v_tok + v2a).squeeze(1)
        a_out = self.norm_a(a_tok + a2v).squeeze(1)
        fused = (v_out + a_out) / 2.0
        logits = self.classifier(fused)
        return logits, v_out, a_out

# ---------------- Full AV Model ----------------
class AVFusionModel(nn.Module):
    def __init__(self, modelA, modelB, fusion_head):
        super().__init__()
        self.modelA = modelA
        self.modelB = modelB
        self.fusion = fusion_head

    def forward(self, frames, mels):
        v = self.modelA(frames)
        a = self.modelB(mels)
        logits, v_emb, a_emb = self.fusion(v, a)
        return logits, v_emb, a_emb

# ---------------- Build ----------------
modelA = ModelA_Visual(ckpt_path=MODEL_A_CKPT, device=DEVICE, embed_dim=256).to(DEVICE)
modelB = ModelB_Audio(ckpt_path=MODEL_B_CKPT, device=DEVICE, embed_dim=256).to(DEVICE)
fusion_head = CrossAttentionFusion(dim=256, heads=4).to(DEVICE)
avmodel = AVFusionModel(modelA, modelB, fusion_head).to(DEVICE)


# ---------------- Freeze schedule ----------------
def set_requires_grad(module, requires):
    for p in module.parameters():
        p.requires_grad = requires

# Freeze encoders initially
set_requires_grad(modelA, False)
set_requires_grad(modelB, False)
# fusion head trainable
set_requires_grad(fusion_head, True)

# ---------------- Losses / Optimizers ----------------
# compute class weights
vals, counts = np.unique(labels, return_counts=True)
cw = {int(v): len(labels)/c for v,c in zip(vals,counts)}
weight_tensor = torch.tensor([cw.get(0,1.0), cw.get(1,1.0)], dtype=torch.float).to(DEVICE)
ce_loss_fn = nn.CrossEntropyLoss(weight=weight_tensor)

# optimizer: different lrs for fusion and encoders (encoders frozen initially)
opt = torch.optim.AdamW([
    {"params": fusion_head.parameters(), "lr": LR_FUSION},
    {"params": modelA.parameters(), "lr": LR_ENCODER},
    {"params": modelB.parameters(), "lr": LR_ENCODER},
], weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

# ---------------- Contrastive loss helper ----------------
def contrastive_loss(v_emb, a_emb, labels, margin=0.5):
    # v_emb, a_emb: [B,dim]
    cos = F.cosine_similarity(v_emb, a_emb, dim=1)  # [-1,1]
    pos_mask = (labels == 0).float()    # real -> want high similarity
    neg_mask = (labels == 1).float()    # fake -> want low similarity
    pos_loss = pos_mask * (1.0 - cos)
    neg_loss = neg_mask * F.relu(cos - margin)
    loss = (pos_loss + neg_loss).mean()
    return loss

# ---------------- Train / Eval ----------------
def train_one_epoch(epoch):
    avmodel.train()
    running_loss = 0.0
    running_ce = 0.0
    running_con = 0.0
    total = 0
    correct = 0
    for i, (frames, mels, labels_batch) in enumerate(tqdm(train_loader, desc=f"Train E{epoch}")):
        frames = frames.to(DEVICE).float()
        mels = mels.to(DEVICE).float()
        labels_batch = labels_batch.to(DEVICE)
        # Optional: random feature interchange augmentation (swap audio in some samples)
        if random.random() < 0.15:
            # swap audio between two random items in batch
            j = torch.randperm(frames.size(0))
            mels = mels[j]

        opt.zero_grad()
        logits, v_emb, a_emb = avmodel(frames, mels)
        ce = ce_loss_fn(logits, labels_batch)
        con = contrastive_loss(v_emb, a_emb, labels_batch)
        loss = ce + CONTRASTIVE_WEIGHT * con
        loss.backward()
        opt.step()

        running_loss += float(loss.item()) * labels_batch.size(0)
        running_ce += float(ce.item()) * labels_batch.size(0)
        running_con += float(con.item()) * labels_batch.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels_batch).sum().item()
        total += labels_batch.size(0)

        if (i+1) % PRINT_EVERY == 0:
            print(f"[E{epoch}] iter {i+1} loss {running_loss/total:.4f} ce {running_ce/total:.4f} con {running_con/total:.4f}")

    acc = correct / max(1, total)
    avg_loss = running_loss / max(1, total)
    return avg_loss, acc

@torch.no_grad()
def validate(epoch):
    avmodel.eval()
    ys, yps, probs = [], [], []
    total, correct = 0, 0
    for frames, mels, labels_batch in tqdm(val_loader, desc=f"Val E{epoch}"):
        frames = frames.to(DEVICE).float()
        mels = mels.to(DEVICE).float()
        labels_batch = labels_batch.to(DEVICE)
        logits, v_emb, a_emb = avmodel(frames, mels)
        prob = F.softmax(logits, dim=1)[:,1]
        preds = logits.argmax(dim=1)
        ys.extend(labels_batch.cpu().numpy().tolist())
        yps.extend(preds.cpu().numpy().tolist())
        probs.extend(prob.cpu().numpy().tolist())
        correct += (preds == labels_batch).sum().item()
        total += labels_batch.size(0)
    acc = correct / max(1, total)
    return np.array(ys), np.array(yps), np.array(probs), acc

# ---------------- Training loop ----------------
best_acc = -1.0
print("Start training on device:", DEVICE)

for epoch in range(1, EPOCHS + 1):
    
    if epoch == FREEZE_EPOCHS + 1:
        print(f"[SCHEDULE] Unfreezing encoders at epoch {epoch}")
        set_requires_grad(modelA, True)
        set_requires_grad(modelB, True)
        set_requires_grad(fusion_head, True)
        opt = torch.optim.AdamW([
            {"params": fusion_head.parameters(), "lr": LR_FUSION},
            {"params": modelA.parameters(), "lr": LR_ENCODER},
            {"params": modelB.parameters(), "lr": LR_ENCODER},
        ], weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, EPOCHS - epoch + 1))

    train_loss, train_acc = train_one_epoch(epoch)
    ys, yps, probs, val_acc = validate(epoch)
    scheduler.step()

    print(f"Epoch {epoch}/{EPOCHS} — train_loss {train_loss:.4f} train_acc {train_acc:.4f} val_acc {val_acc:.4f}")

    # save best
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({
            "epoch": epoch,
            "model_state": avmodel.state_dict(),
            "modelA_state": modelA.state_dict(),
            "modelB_state": modelB.state_dict(),
            "fusion_state": fusion_head.state_dict(),
            "optimizer": opt.state_dict(),
            "best_acc": best_acc
        }, OUTPUT_DIR / "modelC_crossfusion_best.pth")
        print(f"Saved best model (val_acc {best_acc:.4f}) to {OUTPUT_DIR/'modelC_crossfusion_best.pth'}")

    # periodic save
    if epoch % 5 == 0:
        torch.save({
            "epoch": epoch,
            "model_state": avmodel.state_dict(),
            "optimizer": opt.state_dict(),
        }, OUTPUT_DIR / f"modelC_crossfusion_epoch{epoch}.pth")

# ---------------- Final evaluation & summary ----------------
print("Final evaluation on validation set...")
ys, yps, probs, val_acc = validate("final")
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
print("Val acc:", val_acc)
print(classification_report(ys, yps, target_names=["Real","Fake"], digits=4))
cm = confusion_matrix(ys, yps)
print("Confusion matrix:\n", cm)
try:
    auc = roc_auc_score(ys, probs)
    print("ROC AUC:", auc)
except Exception as e:
    print("Could not compute AUC:", e)

# save predictions
pd.DataFrame({"true": ys, "pred": yps, "prob_fake": probs}).to_csv(OUTPUT_DIR / "predictions_val.csv", index=False)
print("Saved predictions to", OUTPUT_DIR / "predictions_val.csv")
print("Training finished. Best val acc:", best_acc)


Loading dataset...
Samples: 21544 Label counts: {1: 21044, 0: 500}


C:\Users\admin\AppData\Local\Temp\ipykernel_30336\3996871257.py:192: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location=device)


[INFO] ModelA loaded pretrained CNN/Transformer weights from checkpoints/ckpt_epoch_10.pth
[WARN] Could not load ModelB ckpt: Weights only load failed. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
Please file an issue with the following so that we can make `weights_only=True` compatible with your use case: WeightsUnpickler error: Can only build Tensor, Parameter, OrderedDict or types allowlisted via `add_safe_globals`, but got <class 'numpy.dtypes.Float64DType'>

Check the documentation of torch.load to learn more about types accepted by default with weights_only https://pytorch.org/docs/stable/generated/torch.load.html.
Start training on device: cuda


Train E1:   2%|█▍                                                                    | 50/2424 [00:34<26:02,  1.52it/s]

[E1] iter 50 loss 0.3630 ce 0.2211 con 0.4055


...

[E15] iter 2400 loss 0.0062 ce 0.0049 con 0.0039


Val E15: 100%|███████████████████████████████████████████████████████████████████████| 270/270 [03:36<00:00,  1.25it/s]


Epoch 15/20 — train_loss 0.0062 train_acc 0.9874 val_acc 0.9763


Train E16:   2%|█▍                                                                 | 50/2424 [03:35<2:56:32,  4.46s/it]

[E16] iter 50 loss 0.0031 ce 0.0018 con 0.0037


In [ ]:
#sync_sensitivity_analysis

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, roc_auc_score

def evaluate_sync_sensitivity(model, test_loader, device, time_shifts_ms=[-1000, -500, -200, -100, -50, 0, 50, 100, 200, 500, 1000]):
    """
    Evaluates Model C performance under audio-visual temporal desynchronization.
    - time_shifts_ms: Temporal shift applied to audio in milliseconds.
    """
    model.eval()
    results = []
    
    # Hop length = 160 @ 16kHz -> 1 frame = 10ms
    hop_ms = 10.0 

    for shift in time_shifts_ms:
        frame_shift = int(shift / hop_ms)
        all_labels = []
        all_preds = []
        all_probs = []

        with torch.no_grad():
            for frames, mels, labels in test_loader:
                frames = frames.to(device).float()
                mels = mels.to(device).float()
                labels = labels.to(device)

                # Apply temporal shift to Mel-spectrogram tensor [B, 1, 80, T]
                if frame_shift != 0:
                    B, C, F_dim, T_dim = mels.shape
                    shifted_mels = torch.zeros_like(mels)
                    if frame_shift > 0:
                        # Audio delayed relative to video
                        if frame_shift < T_dim:
                            shifted_mels[:, :, :, frame_shift:] = mels[:, :, :, :-frame_shift]
                    else:
                        # Audio advanced relative to video
                        abs_shift = abs(frame_shift)
                        if abs_shift < T_dim:
                            shifted_mels[:, :, :, :-abs_shift] = mels[:, :, :, abs_shift:]
                    mels = shifted_mels

                # Forward pass through Cross-Attention Model
                logits, v_emb, a_emb = model(frames, mels)
                probs = F.softmax(logits, dim=1)[:, 1]
                preds = logits.argmax(dim=1)

                all_labels.extend(labels.cpu().numpy())
                all_preds.extend(preds.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())

        acc = accuracy_score(all_labels, all_preds)
        try:
            auc = roc_auc_score(all_labels, all_probs)
        except Exception:
            auc = 0.5

        results.append({
            "shift_ms": shift,
            "accuracy": acc,
            "auc": auc
        })
        print(f"Shift: {shift:>5} ms | Accuracy: {acc:.4f} | AUC: {auc:.4f}")

    return pd.DataFrame(results)

In [ ]:
# Run evaluation across shifts
df_sync = evaluate_sync_sensitivity(avmodel, val_loader, DEVICE)

# Save evaluation metric log for manuscript verification
#df_sync.to_csv("modelC_option3c_crossfusion/sync_sensitivity_results.csv", index=False)

# Generate publication-quality figure
plt.figure(figsize=(7, 4.5), dpi=300)
plt.plot(df_sync["shift_ms"], df_sync["accuracy"], marker='o', linewidth=2, label="Accuracy")
plt.plot(df_sync["shift_ms"], df_sync["auc"], marker='s', linewidth=2, linestyle='--', label="ROC-AUC")

plt.axvline(x=0, color='gray', linestyle=':', alpha=0.7, label="Synchronized (0 ms)")
plt.title("Audio-Visual Synchronization Sensitivity Analysis (DFusionNet)", fontsize=11)
plt.xlabel("Audio Temporal Offset $\Delta t$ (ms)", fontsize=10)
plt.ylabel("Detection Performance", fontsize=10)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("modelC_option3c_crossfusion/sync_sensitivity_analysis.png")
